In [115]:
import pandas as pd
import json

### Funções auxiliares

In [116]:
def converter_datas(serie: pd.Series) -> pd.Series:
    """Converte datas nos formatos ISO (YYYY-MM-DD) e BR (DD/MM/YYYY) para um único formato"""
    iso = pd.to_datetime(serie, format='%Y-%m-%d', errors='coerce')
    br = pd.to_datetime(serie, format='%d/%m/%Y', errors='coerce')

    resultado = iso.fillna(br)

    nao_parseadas = resultado.isna().sum()
    ## if nao_parseadas > 0:
        ## log...

    return resultado

In [117]:
def converter_moeda_br(serie: pd.Series) -> pd.Series:
    """Converte valores em reais (R$ 1.250,00) para o formato do Pandas (1250.00) e converte para float"""
    return (serie
            .str.replace(r'R\$\s*', '', regex=True)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .str.strip()
            .astype(float))

In [118]:
def converter_percentual(serie: pd.Series) -> pd.Series:
    """Converte percentuais (%) para decimal"""
    return ((serie
 .str.replace('%', '', regex=False)
 .str.replace(',', '.', regex=False)
 .str.strip()
 .astype(float)
 .div(100)))

In [119]:
def limpar_colunas_string(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Remove espaços duplos em colunas que são strings e aplica strip()"""
    for col in colunas:
        df[col] = (df[col]
                   .str.replace(r'\s+', ' ', regex=True)
                   .str.strip())
    
    return df

### Leitura dos arquivos

In [120]:
vendas_jan2025 = pd.read_csv('vendas_jan2025.csv', sep=';', encoding='latin-1')

with open('vendas_fev2025.json', 'r', encoding='utf-8') as f:
    vendas_fev2025 = pd.json_normalize(json.load(f), sep='_')

vendas_mar2025 = pd.read_excel('vendas_mar2025.xlsx', sheet_name='dados')

lojas_nordeste = pd.read_excel('lojas_nordeste.xlsx', sheet_name='ativas')

metas_q1_2025 = pd.read_csv('metas_q1_2025.csv', sep=',', encoding='utf-8')

### Limpeza dos arquivos

In [121]:
vendas_jan2025 = vendas_jan2025.drop_duplicates()

vendas_jan2025['data_venda'] = converter_datas(vendas_jan2025['data_venda'])
vendas_jan2025['preco_unitario'] = converter_moeda_br(vendas_jan2025['preco_unitario'])
vendas_jan2025['desconto_percentual'] = converter_percentual(vendas_jan2025['desconto_percentual'])

colunas_string = vendas_jan2025.select_dtypes('string').columns
vendas_jan2025 = limpar_colunas_string(vendas_jan2025, colunas_string)
vendas_jan2025['vendedor'] = vendas_jan2025['vendedor'].str.title()

vendas_jan2025 = vendas_jan2025.reset_index(drop=True)


In [122]:
vendas_fev2025 = vendas_fev2025.drop_duplicates()

vendas_fev2025['data_venda'] = pd.to_datetime(vendas_fev2025['data_venda'], format='%d-%m-%Y')
vendas_fev2025['desconto_percentual'] = converter_percentual(vendas_fev2025['desconto_percentual'])

colunas_string = vendas_fev2025.select_dtypes('str').columns
vendas_fev2025 = limpar_colunas_string(vendas_fev2025, colunas_string)
vendas_fev2025['vendedor'] = vendas_fev2025['vendedor'].str.title()

vendas_fev2025 = vendas_fev2025.reset_index(drop=True)

In [123]:
vendas_mar2025 = vendas_mar2025.drop_duplicates()

vendas_mar2025['preco_unitario'] = converter_moeda_br(vendas_mar2025['preco_unitario'])
vendas_mar2025['desconto_percentual'] = (vendas_mar2025['desconto_percentual'].div(100))

colunas_strings = vendas_mar2025.select_dtypes('str').columns
vendas_mar2025 = limpar_colunas_string(vendas_mar2025, colunas_strings)
vendas_mar2025['vendedor'] = vendas_mar2025['vendedor'].str.title()

vendas_mar2025 = vendas_mar2025.reset_index(drop=True)

In [124]:
lojas_nordeste = lojas_nordeste.drop_duplicates()

lojas_nordeste['data_inauguracao'] = converter_datas(lojas_nordeste['data_inauguracao'])

colunas_strings = lojas_nordeste.select_dtypes('str').columns
lojas_nordeste = limpar_colunas_string(lojas_nordeste, colunas_strings)

lojas_nordeste = lojas_nordeste.reset_index(drop=True)

In [125]:
metas_q1_2025 = metas_q1_2025.drop_duplicates()

metas_q1_2025['meta_faturamento_q1'] = converter_moeda_br(metas_q1_2025['meta_faturamento_q1'])

metas_q1_2025 = metas_q1_2025.reset_index(drop=True)

### Merge dos Dataframes

In [126]:
vendas_trimestre = pd.concat([vendas_jan2025, vendas_fev2025, vendas_mar2025], axis=0, ignore_index=True, verify_integrity=True)

In [136]:
vendas_trimestre = (vendas_trimestre.merge(right=lojas_nordeste, how='inner', on='id_loja')).merge(right=metas_q1_2025, how='inner', on='id_loja')

In [137]:
vendas_trimestre

,id_transacao,data_venda,id_loja,id_produto,quantidade,preco_unitario,desconto_percentual,vendedor,forma_pagamento,nome_loja,cidade,estado,metros_quadrados,data_inauguracao,gerente,meta_faturamento_q1,meta_transacoes_q1
0,7079,2025-01-07,103,1006,53,109.32,0.010,Carlos Souza,Cartão Crédito,Frutally Salvador Pituba,Salvador,BA,NaN,2021-11-05,Bruno Nascimento,310000.0,950
1,7021,2025-01-13,101,1012,37,72.16,0.109,Carlos Souza,PIX,Frutally Fortaleza Centro,Fortaleza,CE,320.0,2019-03-12,Ricardo Mendes,250000.0,800
2,7069,2025-01-01,105,1011,59,107.24,0.118,João Pereira,Dinheiro,Frutally São Luís,São Luís,MA,410.0,2023-08-15,Thiago Borges,200000.0,700
3,7052,2025-01-22,103,1004,15,75.05,0.183,Carlos Souza,PIX,Frutally Salvador Pituba,Salvador,BA,NaN,2021-11-05,Bruno Nascimento,310000.0,950
4,7055,2025-01-21,104,1001,13,76.65,0.191,NaN,Boleto,Frutally Teresina Centro,Teresina,PI,195.0,2022-01-10,Luciana Rocha,120000.0,450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392,9078,2025-03-19,102,1007,11,37.45,0.135,Maria Lima,Cartão Débito,Frutally Recife Boa Viagem,Recife,PE,280.0,2020-07-15,Patrícia Alves,180000.0,600
393,9075,2025-03-07,101,1001,7,98.79,0.163,NaN,Cartão Crédito,Frutally Fortaleza Centro,Fortaleza,CE,320.0,2019-03-12,Ricardo Mendes,250000.0,800
394,9104,2025-03-29,105,1013,25,21.49,0.184,Maria Lima,PIX,Frutally São Luís,São Luís,MA,410.0,2023-08-15,Thiago Borges,200000.0,700
395,9089,2025-03-19,105,1006,55,54.40,0.162,Maria Lima,Cartão Crédito,Frutally São Luís,São Luís,MA,410.0,2023-08-15,Thiago Borges,200000.0,700


In [132]:
lojas_nordeste

,id_loja,nome_loja,cidade,estado,metros_quadrados,data_inauguracao,gerente
0,101,Frutally Fortaleza Centro,Fortaleza,CE,320.0,2019-03-12,Ricardo Mendes
1,102,Frutally Recife Boa Viagem,Recife,PE,280.0,2020-07-15,Patrícia Alves
2,103,Frutally Salvador Pituba,Salvador,BA,NaN,2021-11-05,Bruno Nascimento
3,104,Frutally Teresina Centro,Teresina,PI,195.0,2022-01-10,Luciana Rocha
4,105,Frutally São Luís,São Luís,MA,410.0,2023-08-15,Thiago Borges


In [133]:
metas_q1_2025

,id_loja,meta_faturamento_q1,meta_transacoes_q1
0,101,250000.0,800
1,102,180000.0,600
2,103,310000.0,950
3,104,120000.0,450
4,105,200000.0,700
